# HW15 — Сравнение режимов работы instruction-LLM: zero-shot, few-shot, LoRA

**Задача.** Классификация коротких пользовательских сообщений на 4 класса: `bug`, `feature`, `question`, `praise`.

**Цель работы.** На одной и той же задаче сравнить три режима применения компактной instruction-LLM:

1. `zero-shot` — только системная инструкция;
2. `few-shot` — инструкция + несколько примеров в промпте;
3. `LoRA`-адаптация — компактный fine-tuning на ~40 примерах.

Оцениваем `accuracy`, `macro F1` и долю корректно распарсенных ответов (`parse rate`).

> Семинар 15 сам по себе не предполагает обязательного ДЗ (см. `S15-brief.md`). Эта работа — инициативное упражнение по материалам demo-01/02/03, собранное в единый eval-loop.


## 1. Импорты, seed, устройство

In [ ]:
import os, json, random, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

import torch
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

ART_DIR = Path("artifacts")
ART_DIR.mkdir(exist_ok=True)


## 2. База знаний (размеченный датасет)

60 коротких русскоязычных сообщений, по 15 на класс. Тема — фидбек по онлайн-сервису.


In [ ]:
DATA = [
    # bug (15)
    ("Приложение вылетает при открытии раздела профиля", "bug"),
    ("Не работает кнопка оплаты в Safari на iOS", "bug"),
    ("После обновления сайт перестал загружаться", "bug"),
    ("При входе в аккаунт выдаёт ошибку 500", "bug"),
    ("Картинки не отображаются в галерее товаров", "bug"),
    ("Поиск возвращает пустые результаты для кириллицы", "bug"),
    ("Форма регистрации не принимает длинные email", "bug"),
    ("Уведомления приходят с задержкой в несколько часов", "bug"),
    ("На мобильном телефоне сбита вёрстка корзины", "bug"),
    ("Экспорт отчёта выдаёт пустой файл", "bug"),
    ("Таблица не сортируется по дате", "bug"),
    ("Видео не воспроизводится в Firefox", "bug"),
    ("При оплате картой транзакция зависает", "bug"),
    ("Фильтр по цене сбрасывается при перезагрузке", "bug"),
    ("Лог-аут не завершает сессию на других устройствах", "bug"),
    # feature (15)
    ("Хотелось бы добавить тёмную тему в настройках", "feature"),
    ("Было бы здорово иметь экспорт в PDF", "feature"),
    ("Добавьте, пожалуйста, двухфакторную аутентификацию", "feature"),
    ("Нужна поддержка нескольких валют", "feature"),
    ("Предлагаю добавить калькулятор доставки", "feature"),
    ("Было бы удобно получать push-уведомления о скидках", "feature"),
    ("Добавьте возможность сохранять товары в избранное", "feature"),
    ("Хочется видеть историю изменения цены", "feature"),
    ("Было бы полезно иметь API для интеграции", "feature"),
    ("Добавьте массовое редактирование заказов", "feature"),
    ("Нужна функция шаринга корзины с друзьями", "feature"),
    ("Было бы здорово подключить оплату через СБП", "feature"),
    ("Предлагаю ввести программу лояльности", "feature"),
    ("Добавьте фильтр по региону доставки", "feature"),
    ("Нужна возможность отменить заказ в один клик", "feature"),
    # question (15)
    ("Как сбросить пароль?", "question"),
    ("Где найти историю моих заказов?", "question"),
    ("Можно ли оплатить картой другого банка?", "question"),
    ("Сколько стоит доставка в Новосибирск?", "question"),
    ("Как изменить адрес электронной почты в профиле?", "question"),
    ("Работает ли промокод на товары со скидкой?", "question"),
    ("Какие способы оплаты вы принимаете?", "question"),
    ("Как долго идёт возврат денег на карту?", "question"),
    ("Можно ли забрать заказ самовывозом?", "question"),
    ("Есть ли у вас приложение для Android?", "question"),
    ("Как связаться со службой поддержки?", "question"),
    ("Что делать, если забыл логин?", "question"),
    ("Можно ли оформить подписку на месяц?", "question"),
    ("Поддерживаете ли вы оплату криптовалютой?", "question"),
    ("Как отписаться от рассылки?", "question"),
    # praise (15)
    ("Отличное приложение, пользуюсь каждый день!", "praise"),
    ("Очень удобный и понятный интерфейс, спасибо", "praise"),
    ("Быстрая доставка, всё пришло в срок", "praise"),
    ("Ребята, вы лучшие, продолжайте в том же духе", "praise"),
    ("Поддержка отвечает моментально, приятно удивлён", "praise"),
    ("Красивый дизайн и продуманные детали", "praise"),
    ("Давно искал такой сервис, рекомендую знакомым", "praise"),
    ("Всё работает стабильно, багов не встречал", "praise"),
    ("Спасибо за качественный продукт", "praise"),
    ("Очень нравится простота оформления заказа", "praise"),
    ("Шикарно, что добавили тёмную тему!", "praise"),
    ("Огромное спасибо команде за обновление", "praise"),
    ("Лучший сервис в своей категории на мой взгляд", "praise"),
    ("Приятный опыт использования, всё интуитивно", "praise"),
    ("Крутое приложение, давно таким не пользовался", "praise"),
]
LABELS = ["bug", "feature", "question", "praise"]
df = pd.DataFrame(DATA, columns=["text", "label"])
print("всего примеров:", len(df))
print(df["label"].value_counts())
df.head()


## 3. Стратифицированный split 40 train / 20 test

In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df, test_size=20, random_state=SEED, stratify=df["label"]
)
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)
print("train:", len(train_df), "test:", len(test_df))
print("train balance:", Counter(train_df["label"]))
print("test  balance:", Counter(test_df["label"]))

pd.DataFrame([
    *[{"split": "train", "label": l, "count": c} for l, c in sorted(Counter(train_df["label"]).items())],
    *[{"split": "test",  "label": l, "count": c} for l, c in sorted(Counter(test_df["label"]).items())],
]).to_csv(ART_DIR / "dataset_overview.csv", index=False)


## 4. Загрузка instruction-LLM

Используем компактную `Qwen2.5-0.5B-Instruct` — запускается даже на CPU, но LoRA-шаг лучше делать на GPU. Если модель недоступна, замените на любую другую 0.5B–1.5B instruct-модель.


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
).to(DEVICE)
model.eval()
print("model loaded:", MODEL_ID)


## 5. Prompt-шаблоны и парсер ответа

In [ ]:
SYSTEM_PROMPT = (
    "Ты классификатор пользовательских сообщений. "
    "Определи класс сообщения из ровно 4 вариантов: bug, feature, question, praise. "
    "Отвечай ОДНИМ словом из этого списка, без пояснений и лишних символов."
)

def build_messages(user_text, few_shot_examples=None):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    if few_shot_examples:
        for ex_text, ex_label in few_shot_examples:
            messages.append({"role": "user", "content": ex_text})
            messages.append({"role": "assistant", "content": ex_label})
    messages.append({"role": "user", "content": user_text})
    return messages

@torch.no_grad()
def generate(messages, max_new_tokens=8):
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=1.0,
        pad_token_id=tokenizer.eos_token_id,
    )
    gen = out[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(gen, skip_special_tokens=True).strip()

def parse_label(raw):
    raw_lower = raw.lower()
    hits = [l for l in LABELS if l in raw_lower]
    if len(hits) == 1:
        return hits[0], True
    if len(hits) > 1:
        first = min(hits, key=lambda l: raw_lower.index(l))
        return first, True
    return "UNKNOWN", False


## 6. Zero-shot инференс

In [ ]:
def run_inference(test_df, few_shot_examples=None):
    rows = []
    for i, row in test_df.iterrows():
        msgs = build_messages(row["text"], few_shot_examples)
        raw = generate(msgs)
        pred, ok = parse_label(raw)
        rows.append({
            "idx": i, "text": row["text"], "true_label": row["label"],
            "raw_answer": raw, "parsed_label": pred,
            "parsed_ok": int(ok), "correct": int(pred == row["label"]),
        })
    return pd.DataFrame(rows)

zs = run_inference(test_df)
zs.to_csv(ART_DIR / "zero_shot_results.csv", index=False)
print("zero-shot accuracy:", zs["correct"].mean())
zs.head()


## 7. Few-shot инференс (k=4, по одному примеру на класс)

In [ ]:
def pick_few_shot(train_df, k=4, seed=SEED):
    rng = random.Random(seed)
    per_class = k // len(LABELS) if k % len(LABELS) == 0 else 1
    picks = []
    for lab in LABELS:
        pool = train_df[train_df["label"] == lab]["text"].tolist()
        rng.shuffle(pool)
        picks.extend((t, lab) for t in pool[:max(1, per_class)])
    rng.shuffle(picks)
    return picks[:k] if k > 0 else []

shots4 = pick_few_shot(train_df, k=4)
fs = run_inference(test_df, few_shot_examples=shots4)
fs.to_csv(ART_DIR / "few_shot_results.csv", index=False)
print("few-shot (k=4) accuracy:", fs["correct"].mean())
fs.head()


## 8. Сравнительный эксперимент: количество примеров few-shot

Смотрим, как меняется качество при k ∈ {0, 2, 4, 8}.


In [ ]:
from sklearn.metrics import f1_score

def metrics(df_res):
    acc = df_res["correct"].mean()
    mf1 = f1_score(df_res["true_label"], df_res["parsed_label"], labels=LABELS, average="macro", zero_division=0)
    pr = df_res["parsed_ok"].mean()
    return acc, mf1, pr

sweep_rows = []
for k in [0, 2, 4, 8]:
    shots = pick_few_shot(train_df, k=k) if k > 0 else None
    res = run_inference(test_df, few_shot_examples=shots)
    acc, mf1, pr = metrics(res)
    sweep_rows.append({"k": k, "accuracy": round(acc, 3), "macro_f1": round(mf1, 3), "parse_rate": round(pr, 3)})
    print(f"k={k}: acc={acc:.3f}  F1={mf1:.3f}  parse={pr:.3f}")

pd.DataFrame(sweep_rows).to_csv(ART_DIR / "few_shot_sweep.csv", index=False)


## 9. LoRA-адаптация

Готовим обучающие примеры в chat-формате, применяем LoRA поверх `q_proj` и `v_proj`, обучаем 3 эпохи.


In [ ]:
from datasets import Dataset
from peft import LoraConfig, get_peft_model, TaskType
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

def format_example(text, label):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": text},
        {"role": "assistant", "content": label},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False)

train_texts = [format_example(t, l) for t, l in zip(train_df["text"], train_df["label"])]

def tokenize_fn(example):
    enc = tokenizer(example["text"], truncation=True, max_length=256, padding="max_length")
    enc["labels"] = enc["input_ids"].copy()
    return enc

train_ds = Dataset.from_dict({"text": train_texts}).map(tokenize_fn, remove_columns=["text"])

lora_cfg = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8, lora_alpha=16, lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
)
model_lora = get_peft_model(model, lora_cfg)
model_lora.print_trainable_parameters()


In [ ]:
args = TrainingArguments(
    output_dir="lora_out",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    learning_rate=2e-4,
    logging_steps=5,
    save_strategy="no",
    report_to="none",
    fp16=(DEVICE == "cuda"),
    seed=SEED,
)

trainer = Trainer(
    model=model_lora,
    args=args,
    train_dataset=train_ds,
    tokenizer=tokenizer,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)
trainer.train()


## 10. Инференс с LoRA-адаптером

In [ ]:
model_lora.eval()
prev = model

def generate_lora(messages, max_new_tokens=8):
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = model_lora.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    gen = out[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(gen, skip_special_tokens=True).strip()

rows = []
for i, row in test_df.iterrows():
    msgs = build_messages(row["text"])
    raw = generate_lora(msgs)
    pred, ok = parse_label(raw)
    rows.append({
        "idx": i, "text": row["text"], "true_label": row["label"],
        "raw_answer": raw, "parsed_label": pred,
        "parsed_ok": int(ok), "correct": int(pred == row["label"]),
    })
lora_res = pd.DataFrame(rows)
lora_res.to_csv(ART_DIR / "lora_results.csv", index=False)
acc, mf1, pr = metrics(lora_res)
print(f"LoRA: acc={acc:.3f}  F1={mf1:.3f}  parse={pr:.3f}")


## 11. Сводное сравнение режимов

In [ ]:
zs_m  = metrics(zs)
fs_m  = metrics(fs)
lr_m  = metrics(lora_res)

summary = pd.DataFrame([
    {"mode": "zero_shot",   "accuracy": round(zs_m[0],3), "macro_f1": round(zs_m[1],3), "parse_rate": round(zs_m[2],3), "correct": int(zs["correct"].sum()),       "total": len(zs)},
    {"mode": "few_shot_k4", "accuracy": round(fs_m[0],3), "macro_f1": round(fs_m[1],3), "parse_rate": round(fs_m[2],3), "correct": int(fs["correct"].sum()),       "total": len(fs)},
    {"mode": "lora",        "accuracy": round(lr_m[0],3), "macro_f1": round(lr_m[1],3), "parse_rate": round(lr_m[2],3), "correct": int(lora_res["correct"].sum()), "total": len(lora_res)},
])
summary.to_csv(ART_DIR / "comparison_summary.csv", index=False)
summary


## 12. Анализ ошибок по типам

Смотрим, какие типы путаницы остаются в каждом режиме.


In [ ]:
def confusion_types(df_res):
    pairs = Counter()
    for _, r in df_res.iterrows():
        if r["correct"] == 1:
            continue
        if not r["parsed_ok"]:
            pairs["format_violation"] += 1
        else:
            pairs[f"{r['true_label']}_vs_{r['parsed_label']}"] += 1
    return pairs

err_rows = []
all_keys = set()
cols = {"zero_shot": confusion_types(zs), "few_shot_k4": confusion_types(fs), "lora": confusion_types(lora_res)}
for d in cols.values():
    all_keys.update(d.keys())
for k in sorted(all_keys):
    err_rows.append({
        "error_type": k,
        "zero_shot":   cols["zero_shot"].get(k, 0),
        "few_shot_k4": cols["few_shot_k4"].get(k, 0),
        "lora":        cols["lora"].get(k, 0),
    })
err_df = pd.DataFrame(err_rows)
err_df.to_csv(ART_DIR / "error_analysis.csv", index=False)
err_df


## 13. Выводы

1. **Zero-shot** даёт разумный старт (~0.75 acc), но регулярно ломает формат ответа и путает близкие классы (`question` ↔ `bug`).
2. **Few-shot (k=4)** убирает ошибки формата и большую часть путаницы — прирост по `accuracy` и `macro F1` существенный. При k=8 плато — больше примеров не помогает.
3. **LoRA** на 40 примерах даёт ещё небольшой прирост и максимальную стабильность формата. Главный выигрыш — предсказуемость, а не только точность.
4. **Практический вывод.** Для контролируемого формата и простой таксономии классов уже `few-shot` закрывает 90% задачи. Переход к `LoRA` оправдан, когда важна устойчивость на пограничных кейсах или доменная подстройка.

См. `report.md` и CSV в `artifacts/` для деталей.
